# Lecture 12: Simple Linear Regression

This notebook introduces a one-predictor linear regression workflow using the synthetic `career_outcomes` dataset.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
df = pd.read_csv(DATA / "career_outcomes.csv")
df.head()


## Analytic Question

How much does salary tend to change with additional training hours?

Before fitting a model, state the outcome, predictor, expected direction, and one plausible omitted variable.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=df, x="training_hours", y="salary_k_eur", alpha=0.65, ax=ax)
sns.regplot(data=df, x="training_hours", y="salary_k_eur", scatter=False, color="black", ax=ax)
ax.set(title="Salary and training hours", xlabel="Training hours", ylabel="Salary, thousand EUR")


In [ ]:
model = smf.ols("salary_k_eur ~ training_hours", data=df).fit()
print(model.summary())


In [ ]:
intercept = model.params["Intercept"]
slope = model.params["training_hours"]
print(f"Predicted salary with no training: {intercept:.1f} thousand EUR")
print(f"Estimated salary change per training hour: {slope:.2f} thousand EUR")


In [ ]:
new_workers = pd.DataFrame({"training_hours": [10, 40, 80]})
predictions = model.get_prediction(new_workers).summary_frame(alpha=0.05)
new_workers.join(predictions[["mean", "mean_ci_lower", "mean_ci_upper"]])


In [ ]:
diagnostics = df.assign(fitted=model.fittedvalues, residual=model.resid)
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=diagnostics, x="fitted", y="residual", alpha=0.65, ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set(title="Residuals versus fitted values", xlabel="Fitted salary", ylabel="Residual")


## LLM Check

Prompt an LLM for a plain-language interpretation of the slope. Revise the response so it states association, units, and the limits of causal interpretation.
